# Plant-Based and Other Labels Correctness Check

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import scipy as sp
import statsmodels as sm
import sklearn as sk
import os
import re
import time
from tqdm import tqdm

In [ ]:
%store -r static_reference_data
%store -r restaurant_sales_data

In [ ]:
before_after_details = static_reference_data[0]
customers = static_reference_data[1]
items_tagged = static_reference_data[2]
locations = static_reference_data[3]

In [ ]:
items_tagged['item_type'].value_counts()

In [ ]:
items_tagged['dish_category'].value_counts()

In [ ]:
items_tagged['ingredients'].value_counts()

In [ ]:
items_tagged['brand'].value_counts()

Filter

In [ ]:
items_tagged_noalc = items_tagged[items_tagged['dish_category'] != 'Alcohol']
plant_based = items_tagged_noalc.loc[items_tagged_noalc['is_plant_based'] == 'yes',:]

Odd nonfood item: candle

In [ ]:
non_food = items_tagged.loc[(items_tagged['is_plant_based'] == 'no') & (items_tagged['item_name'].str.lower().str.contains('soy')), :].iloc[-1]

In [ ]:
pd.set_option('display.max_rows', 500)

Ingredient mislabels

In [ ]:
impossible_ingredient_mislabel = items_tagged[items_tagged['brand'] == 'Impossible']
plant_based_ingredient_mislabel = items_tagged[(items_tagged['ingredients'] != 'Unsure') & (items_tagged['is_plant_based'] == 'yes')]

Plant based label false positives

In [ ]:
veg_keywords = ['oat','alternative','vegan','substitute','jackfru','plant based','portobello','marinara','"cheesecake"','dairy free','carrot','(vg)','v - ','-v','v/','melon','quinoa','corn','rhubarb','vegan','buddha','fruit','watermelon','cucumber','oyster mushroom','(ve)','jackfruit','tofu','berry','buffalo cauliflower']
mask = ~plant_based['item_name'].str.lower().str.contains('non dairy')
for veg_keyword in veg_keywords:
    mask = mask & ~plant_based['item_name'].str.lower().str.contains(veg_keyword)
labeled_plant_based_keywords_removed = plant_based[mask]

In [ ]:
gray_area_foods = ['milk','soup','salad','cake','pizza','sandwich','melt','cheese','grits']
mask = labeled_plant_based_keywords_removed['item_name'].str.lower().str.contains('milk')
for food in gray_area_foods:
    mask = mask | labeled_plant_based_keywords_removed['item_name'].str.lower().str.contains(food)
labeled_plant_based_gray_area_df = labeled_plant_based_keywords_removed[mask]

In [ ]:
labeled_plant_based_gray_area_df['dish_category'] = pd.Categorical(labeled_plant_based_gray_area_df['dish_category'], 
                       categories=["Dairy Drink","Coffee & Tea","Cheese and Charcuterie","Cakes","Pancake","Pizza","Potato Appetizers",'Unsure','Salads','Soups & Stews', 'Sandwich', 'Grain and Salad Bowls','Burger','Water','Other Egg Mains', 'Other Desserts','Rice Dishes', 'Wraps', 'Dips', 'Smoothie', 'Stir Fry & Noodles','Vegetable Side'],
                       ordered=True)
clear_false_positives = labeled_plant_based_gray_area_df.sort_values('dish_category').loc[:5654,:]
possible_false_positives = labeled_plant_based_gray_area_df.sort_values('dish_category').loc[105:,:]

Manual plant-based label false positives from sample size of ~600

In [ ]:
manual_sample = pd.read_csv('mislabeling/possible_false_positives_labeled_plant_based.txt', sep="\t", header=None)
print(manual_false_positives.shape)
manual_possible_false_positives = manual_sample.loc[~manual_sample[0].str.contains("##") & manual_false_positives[0].str.contains("#"),:]
print(manual_possible_false_positives.shape)
manual_clear_false_positives = manual_sample.loc[manual_sample[0].str.contains("##"),:]
print(manual_clear_false_positives.shape)
print("Fraction definitely vegan: ", (manual_sample.shape[0] - manual_possible_false_positives.shape[0] - manual_clear_false_positives.shape[0])/manual_sample.shape[0])
print("Fraction definitely not: ", manual_clear_false_positives.shape[0]/manual_sample.shape[0])

In [ ]:
manual_clear_false_positives

In [ ]:
manual_clear_false_positives

Plant-based label false negatives

In [ ]:
clear_false_negatives = items_tagged.loc[(items_tagged['is_plant_based'] == 'no') & (items_tagged['item_name'].str.lower().str.contains('"')), :].loc[[2849,6011]]
possible_false_negatives = items_tagged.loc[(items_tagged['is_plant_based'] == 'no') & (items_tagged['item_name'].str.lower().str.contains('vegan')), :].iloc[2:,:]

Write to file

In [ ]:
with pd.ExcelWriter('mislabeling/mislabels.xlsx') as writer:  
    impossible_ingredient_mislabel.to_excel(writer, sheet_name='Impossible_Ingredients')
    plant_based_ingredient_mislabel.to_excel(writer, sheet_name='Plant_Based_Ingredients')
    clear_false_positives.to_excel(writer, sheet_name='Clear_False_Positives_Plant_Based_Label')
    possible_false_positives.to_excel(writer, sheet_name='Possible_False_Positives_Plant_Based_Label')
    manual_clear_false_positives.to_excel(writer, sheet_name='Manual_Clear_False_Positives_plant_Based_Label')
    manual_possible_false_positives.to_excel(writer, sheet_name='Manual_Possible_False_Positives_plant_Based_Label')
    clear_false_negatives.to_excel(writer, sheet_name='Clear_False_Negative_Plant_Based_Label')
    possible_false_negatives.to_excel(writer, sheet_name='Possible_False_Negative_Plant_Based_Label')
